# Exercise 2 — PaperAccount.sell and trade log

`sell` is the mirror of `buy`: it converts shares back to cash. The trade log (`self.trades`) records every order with full context — date, action, price, shares, cash balance, and portfolio value. This log is what makes paper trading useful for debugging and review.

In [ ]:
import pandas as pd, math
from dataclasses import dataclass, field

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
@dataclass
class Trade:
    date:        object
    action:      str
    price:       float
    shares:      float
    cash_after:  float
    value_after: float
@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:         float = field(init=False)
    shares:       float = field(init=False)
    trades:       list  = field(init=False)

    def __post_init__(self):
        self.cash   = self.initial_cash
        self.shares = 0.0
        self.trades = []

    def portfolio_value(self, price):
        return self.cash + self.shares * float(price)

    def buy(self, date, price, fraction=1.0):
        price = float(price)
        if self.cash <= 0 or price <= 0:
            return None
        shares = (self.cash * fraction) / price
        cost   = shares * price
        if cost > self.cash:
            shares = self.cash / price
            cost   = shares * price
        self.cash   -= cost
        self.shares += shares
        t = Trade(date=date, action="BUY", price=price, shares=shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t

    def sell(self, date, price):
        """Liquidate the entire position.

        Steps:
          1. Guard: if shares <= 0, return None (already flat)
          2. proceeds    = self.shares * price
          3. sold_shares = self.shares
          4. self.cash  += proceeds; self.shares = 0.0
          5. Append Trade(action="SELL", shares=sold_shares, ...)
          6. Return the Trade
        """
        # TODO: ~8 lines
        return None


### Checks

In [ ]:
checks = 0

# 1 — sell when flat returns None
try:
    acc = PaperAccount(10_000.0)
    result = acc.sell("2023-01-01", 100.0)
    assert result is None, f"sell with no shares should return None, got {result}"
    checks += 1; print("✅ 1 sell() with no position returns None")
except Exception as e:
    print("❌ 1:", e)

# 2 — buy then sell: cash fully restored
try:
    acc = PaperAccount(10_000.0)
    acc.buy("2023-01-02",  100.0)
    acc.sell("2023-01-03", 100.0)
    assert abs(acc.shares)             < 1e-6
    assert abs(acc.cash - 10_000.0)   < 1e-6
    checks += 1; print("✅ 2 buy then sell at same price → cash fully restored, shares=0")
except Exception as e:
    print("❌ 2:", e)

# 3 — profit: sell above buy price
try:
    acc = PaperAccount(10_000.0)
    acc.buy("2023-01-02",  100.0)    # 100 shares
    acc.sell("2023-01-10", 110.0)    # sell at +10%
    expected = 100 * 110.0           # 11000
    assert abs(acc.cash - expected)  < 1e-6,         f"expected cash={expected}, got {acc.cash}"
    checks += 1; print("✅ 3 sell at +10% → $11000 cash (profit)")
except Exception as e:
    print("❌ 3:", e)

# 4 — trade log: 2 records after buy+sell
try:
    acc = PaperAccount(10_000.0)
    acc.buy("2023-01-02",  100.0)
    acc.sell("2023-01-10", 120.0)
    assert len(acc.trades) == 2
    assert acc.trades[0].action == "BUY"
    assert acc.trades[1].action == "SELL"
    assert abs(acc.trades[1].cash_after - 12_000.0) < 1e-6
    checks += 1; print("✅ 4 trade log has 2 records: BUY then SELL")
except Exception as e:
    print("❌ 4:", e)

# 5 — sell clears shares (double-sell returns None)
try:
    acc = PaperAccount(10_000.0)
    acc.buy("2023-01-02",  100.0)
    acc.sell("2023-01-10", 100.0)
    result2 = acc.sell("2023-01-11", 100.0)  # already flat
    assert result2 is None, "double-sell should return None"
    assert len(acc.trades) == 2, "double-sell should not add a trade"
    checks += 1; print("✅ 5 second sell on flat position returns None, no extra trade")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
